# 🍃 Coffee Leaf Multi-Task CNN (Lightweight, Mobile/AR-Ready)
## Eksperimen Lengkap: Training, Evaluasi Komparatif, Benchmark Efisiensi, & Ekspor Unity Sentis

Notebook ini menyediakan pipeline pengujian ilmiah lengkap dan transparan untuk skripsi Anda:
1. **Persiapan Lingkungan & GPU T4** di Google Colab.
2. **Penyelarasan Dataset Gabungan** (JMUBEN + JMUBEN V2 + BRACOL) menjadi 5 kelas seimbang.
3. **Training Model 1:** Multi-Task CNN dengan **MobileNetV2** (Shared Backbone + 3 Head: Penyakit, Keparahan, Lokalisasi).
4. **Training Model 2:** Multi-Task CNN dengan **EfficientNet-B0**.
5. **Uji Efisiensi & Komparasi Lengkap (Head-to-Head):**
   - Akurasi & F1-Score Diagnosis Penyakit
   - MAE & RMSE Estimasi Keparahan
   - IoU Bounding Box Lokalisasi AR
   - **Jumlah Parameter Model** (Mana yang lebih sedikit?)
   - **Ukuran Berkas Model (.pth & .onnx)** (Mana yang lebih hemat memori?)
   - **Latensi Inferensi (ms/citra)** (Mana yang lebih cepat dan ringan untuk smartphone?)
6. **Uji Coba Deteksi Citra Tunggal (Inference Test)** dengan visualisasi kotak bounding box.
7. **Ekspor Model ONNX & Metadata JSON** yang siap diimpor ke **Unity Sentis**.

> ⚠️ **PENTING SEBELUM MULAI:**
> Buka menu atas Colab: `Runtime` -> `Change runtime type` -> Pilih **T4 GPU** -> Klik `Save`.

## 1. Hubungkan ke Google Drive & Ekstrak Dataset
Pastikan dataset Anda (`JMUBEN`, `JMUBEN V2`, `BRACOL_coffee_leaf_ images_datasets`) sudah di-zip menjadi satu file (misal `dataset_kopi.zip`) dan di-upload ke Google Drive Anda.

In [ ]:
from google.colab import drive
import os

# 1. Hubungkan Google Drive
drive.mount('/content/drive')

# 2. Ekstrak dataset ke folder /content/ (sesuaikan nama file zip di Drive Anda)
zip_path = '/content/drive/MyDrive/dataset_kopi.zip'

if os.path.exists(zip_path):
    print(f'Mengekstrak {zip_path}...')
    !unzip -q "$zip_path" -d /content/
    print('Ekstraksi selesai!')
else:
    print(f'File {zip_path} tidak ditemukan. Pastikan file zip dataset sudah diupload ke Drive!')

# 3. Cek GPU yang aktif
!nvidia-smi

## 2. Clone Repository GitHub & Masuk ke Folder Classification

In [ ]:
import os

# Clone repo jika belum ada di Colab
if not os.path.exists('/content/coffee-leaf-multitask-cnn'):
    !git clone https://github.com/fairuzzkii/coffee-leaf-multitask-cnn.git /content/coffee-leaf-multitask-cnn
else:
    print('Repository sudah ada, memperbarui ke commit terbaru...')
    %cd /content/coffee-leaf-multitask-cnn
    !git pull

# Pindah ke folder classification
%cd /content/coffee-leaf-multitask-cnn/classification
!pwd

# Install dependensi ONNX
!pip install -q onnx onnxscript onnxruntime

import torch
print('PyTorch Version :', torch.__version__)
print('CUDA Tersedia   :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Perangkat GPU   :', torch.cuda.get_device_name(0))
else:
    print('PERINGATAN: GPU belum aktif! Ubah di menu Runtime -> Change runtime type -> T4 GPU')

## 3. Siapkan Dataset Gabungan (JMUBEN + JMUBEN V2 + BRACOL)
Cell ini membaca folder citra lokal di Colab, menyelaraskan 5 kelas penyakit, dan membaginya ke 70% Train, 15% Val, dan 15% Test bebas data leakage.

In [ ]:
from prepare_combined_dataset import prepare_combined_dataset

# Membuat dataset terpadu seimbang (1.000 gambar per kelas = 5.000 gambar)
# agar proses training di GPU Colab sangat cepat (~5-10 menit per model)
prepare_combined_dataset(
    workspace_root="/content",
    output_csv="dataset/dataset_combined_balanced.csv",
    sample_per_class=1000
)

## 4. Training Model 1: MobileNetV2 (Multi-Task 3-Head)
Melatih MobileNetV2 dengan bobot loss seimbang ($\lambda_1=1.0, \lambda_2=1.0, \lambda_3=1.0$) sebanyak 5 epoch di GPU.

In [ ]:
!python main.py --train \
  --backbone mobilenet \
  --mode multi \
  --dataset combined \
  --csv_file dataset/dataset_combined_balanced.csv \
  --epochs 5 \
  --batch_size 32 \
  --lr 0.01 \
  --lambda1 1.0 \
  --lambda2 1.0 \
  --lambda3 1.0 \
  --experiment_name exp_multitask_mobilenet

## 5. Training Model 2: EfficientNet-B0 (Multi-Task 3-Head)
Melatih EfficientNet-B0 dengan dataset, split, epoch, dan hyperparameter yang identik untuk perbandingan yang adil.

In [ ]:
!python main.py --train \
  --backbone efficientnet \
  --mode multi \
  --dataset combined \
  --csv_file dataset/dataset_combined_balanced.csv \
  --epochs 5 \
  --batch_size 32 \
  --lr 0.01 \
  --lambda1 1.0 \
  --lambda2 1.0 \
  --lambda3 1.0 \
  --experiment_name exp_multitask_efficientnet

## 6. Uji Komparasi & Efisiensi: Lebih Ringan & Cepat Mana?
Cell ini menghitung dan membandingkan secara transparan metrik ilmiah untuk Bab 4 skripsi:
1. **Jumlah Parameter**
2. **Ukuran File Model (.pth dan .onnx)**
3. **Kecepatan Inferensi (Latensi ms per gambar)**
4. **Akurasi & F1-Score Diagnosis Penyakit**
5. **Error Estimasi Keparahan (MAE & RMSE)**
6. **Kecocokan Bounding Box AR (Mean IoU)**

In [ ]:
import os
import json
import time
import torch
import pandas as pd
from architectures.multitask_model import build_model

# Pastikan kedua model sudah diekspor ke ONNX untuk diukur ukuran file-nya
!python export_onnx.py --weights results/exp_multitask_mobilenet/best_weights.pth --backbone mobilenet --output results/model_unity_mobilenet.onnx
!python export_onnx.py --weights results/exp_multitask_efficientnet/best_weights.pth --backbone efficientnet --output results/model_unity_efficientnet.onnx

def benchmark_model(backbone, weights_path, onnx_path, log_path):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # 1. Parameter Model
    model = build_model(backbone=backbone, mode='multi', pretrained=False)
    if os.path.exists(weights_path):
        model.load_state_dict(torch.load(weights_path, map_location=device))
    model.to(device)
    model.eval()
    
    total_params = sum(p.numel() for p in model.parameters())
    
    # 2. Ukuran File
    pth_size_mb = os.path.getsize(weights_path) / (1024 * 1024) if os.path.exists(weights_path) else 0.0
    onnx_size_mb = os.path.getsize(onnx_path) / (1024 * 1024) if os.path.exists(onnx_path) else 0.0
    
    # 3. Latensi Inferensi di GPU (100 iterasi benchmark)
    dummy_input = torch.randn(1, 3, 224, 224, device=device)
    # Warm-up
    for _ in range(10):
        _ = model(dummy_input)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        
    times = []
    for _ in range(100):
        t0 = time.time()
        _ = model(dummy_input)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        times.append((time.time() - t0) * 1000.0) # ms
    
    avg_latency = sum(times) / len(times)
    fps = 1000.0 / avg_latency if avg_latency > 0 else 0.0
    
    # 4. Baca Metrik Evaluasi dari Log Training
    dis_acc, dis_f1, sev_mae, bbox_iou = 0.0, 0.0, 0.0, 0.0
    if os.path.exists(log_path):
        with open(log_path, 'r') as fp:
            data = json.load(fp)
            test_res = data.get('test_results', {})
            dis_acc = test_res.get('disease_acc', 0.0)
            dis_f1 = test_res.get('disease_f1', 0.0)
            sev_mae = test_res.get('severity_mae', 0.0)
            bbox_iou = test_res.get('bbox_iou', 0.0)
            
    return {
        'Backbone': 'MobileNetV2' if backbone == 'mobilenet' else 'EfficientNet-B0',
        'Total Parameters': f'{total_params:,}',
        'Ukuran Bobot (.pth)': f'{pth_size_mb:.2f} MB',
        'Ukuran ONNX (Unity)': f'{onnx_size_mb:.2f} MB',
        'Latensi Inferensi': f'{avg_latency:.2f} ms',
        'Throughput (FPS)': f'{fps:.1f} FPS',
        'Akurasi Penyakit': f'{dis_acc:.2f}%',
        'Macro F1-Score': f'{dis_f1:.2f}%',
        'Severity MAE': f'{sev_mae:.4f}',
        'Bbox Mean IoU': f'{bbox_iou * 100:.2f}%'
    }

# Jalankan benchmark kedua model
res_mobilenet = benchmark_model(
    'mobilenet',
    'results/exp_multitask_mobilenet/best_weights.pth',
    'results/model_unity_mobilenet.onnx',
    'results/exp_multitask_mobilenet/train_logs.json'
)
res_efficientnet = benchmark_model(
    'efficientnet',
    'results/exp_multitask_efficientnet/best_weights.pth',
    'results/model_unity_efficientnet.onnx',
    'results/exp_multitask_efficientnet/train_logs.json'
)

# Tampilkan Tabel Perbandingan Komprehensif
df_comparison = pd.DataFrame([res_mobilenet, res_efficientnet])
print('='*75)
print('TABEL HASIL PERBANDINGAN KOMPARATIF SKRIPSI (HEAD-TO-HEAD)')
print('='*75)
display(df_comparison)

# Simpan tabel ke file CSV agar bisa langsung dicopy ke naskah skripsi
df_comparison.to_csv('results/tabel_perbandingan_skripsi.csv', index=False)
print('\nTabel perbandingan berhasil disimpan ke: results/tabel_perbandingan_skripsi.csv')

## 7. Testing Deteksi Citra Tunggal & Simulasi Output Unity AR
Menguji model pada sampel gambar daun kopi dan memvisualisasikan prediksi penyakit, persentase keparahan, serta kotak bounding box AR.

In [ ]:
import glob
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from simulate_unity_inference import simulate_unity_pipeline

# Cari salah satu gambar sampel di dataset
sample_images = glob.glob('/content/JMUBEN/*/*.*') + glob.glob('/content/JMUBEN V2/*/*.*')
test_img_path = sample_images[0] if len(sample_images) > 0 else 'test.png'

print(f'Menguji citra: {test_img_path}\n')

# Jalankan simulasi inferensi pipeline Unity
prediction = simulate_unity_pipeline(
    image_path=test_img_path,
    onnx_path='results/model_unity_mobilenet.onnx',
    meta_path='results/model_unity_mobilenet_meta.json'
)

# Visualisasi Gambar + Bounding Box Prediksi AR
if prediction:
    img = Image.open(test_img_path).convert('RGB')
    w, h = img.size
    
    bbox = prediction['localization']['model_coordinates_top_left_origin']
    cx, cy, bw, bh = bbox['center_x'] * w, bbox['center_y'] * h, bbox['width'] * w, bbox['height'] * h
    x1, y1 = cx - bw / 2, cy - bh / 2
    
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(img)
    rect = patches.Rectangle((x1, y1), bw, bh, linewidth=2.5, edgecolor='lime', facecolor='none')
    ax.add_patch(rect)
    
    title_text = f"{prediction['diagnosis']['disease_name']} ({prediction['diagnosis']['confidence']*100:.1f}%) | Sev: {prediction['severity']['percentage']}"
    plt.title(title_text, fontsize=12, color='darkgreen', fontweight='bold')
    plt.axis('off')
    plt.show()

## 8. Verifikasi Kesesuaian Numerik ONNX vs PyTorch

In [ ]:
# Verifikasi model MobileNetV2
!python verify_export.py --onnx results/model_unity_mobilenet.onnx --backbone mobilenet

# Verifikasi model EfficientNet-B0
!python verify_export.py --onnx results/model_unity_efficientnet.onnx --backbone efficientnet

## 9. Download Model ONNX & Hasil Eksperimen Skripsi
Download model terbaik beserta file metadata dan tabel komparasi langsung ke komputer lokal Anda.

In [ ]:
from google.colab import files

# 1. Download Model ONNX untuk Unity Sentis
if os.path.exists('results/model_unity_mobilenet.onnx'):
    files.download('results/model_unity_mobilenet.onnx')
    files.download('results/model_unity_mobilenet_meta.json')

# 2. Download Tabel Perbandingan Skripsi (.csv) & Log Training
if os.path.exists('results/tabel_perbandingan_skripsi.csv'):
    files.download('results/tabel_perbandingan_skripsi.csv')
    
if os.path.exists('results/exp_multitask_mobilenet/train_logs.json'):
    files.download('results/exp_multitask_mobilenet/train_logs.json')
    
if os.path.exists('results/exp_multitask_efficientnet/train_logs.json'):
    files.download('results/exp_multitask_efficientnet/train_logs.json')

print('Semua berkas luaran berhasil disiapkan untuk di-download!')